In [2]:
# import os

# os.chdir(r"c:\Users\alber\Desktop\MGR\main_work")

In [1]:
import pandas as pd
# import pathlib
# import fnmatch
# from src.data.data import get_df
# from src.data.wa_converter import prepare_data

In [4]:
res_df = pd.read_csv("/storage2/albert/iSTS-mgr/headlines_score_results.csv")

In [7]:
res_df[res_df['preds'].isna()]

,Unnamed: 0,text_source,text_translation,alignment,score,preds,response
30,30,China stocks,Chinese stocks,EQUI,5,NaN,To determine the appropriate label
71,71,12,10,SIMI,4,NaN,To determine the appropriate label
96,96,in a coma,in a coma,EQUI,5,NaN,To determine the correct label
102,102,in talks,in talks,EQUI,5,NaN,To determine the label for
190,190,leads,leads,EQUI,5,NaN,To determine the appropriate label
...,...,...,...,...,...,...,...
3790,3790,injured,injured,EQUI,5,NaN,To determine the appropriate label
3804,3804,before parliamentary polls,new Cabinet,REL,2,NaN,??\nTo determine the
3848,3848,-not aligned-,killed,NOALI,0,NaN,\n\nTo determine the appropriate
3892,3892,10,39,SIMI,2,NaN,###\n\nTo determine the


In [4]:
ROOT_DIR = os.getcwd()

In [5]:
DATA_PATH = pathlib.Path(ROOT_DIR + "/Data/")
TRAIN_DATA_PATH = DATA_PATH / "train_data"

In [6]:
DATASET = ['answers-students', 'headlines', 'images']

In [7]:
RAW_DATA = DATA_PATH / 'raw'
CHUNKED_DATA  = DATA_PATH / 'chunked'
WA_DATA = DATA_PATH / 'wa'

In [8]:
def find_files_with_phrase(directory, phrase):
    matching_files = []
    # Traverse the directory tree
    for root, dirs, files in os.walk(directory):
        for file in files:
            # Check if the phrase is in the filename
            if fnmatch.fnmatch(file, f'*{phrase}*'):
                matching_files.append(os.path.join(root, file))
    
    return matching_files

In [9]:
def read_raw_file(filename):

    with open(filename) as f:
        texts = []    
        for line in f:
            texts.append(line.strip())
    
    return texts

In [18]:
def get_dataframe(dataset_name, source):
    files = find_files_with_phrase(TRAIN_DATA_PATH / source, dataset_name)
    assert len(files) < 3 or len(files) == 0, "Wrong files"
    
    df = pd.DataFrame()
    
    if source in ["raw", "chunked"]:
        for i, file in enumerate(files):
            df[f'sent{i+1}'] = read_raw_file(file)

    else:
        prepare_data_path = prepare_data(files[0])
        df = get_df_from_wa(prepare_data_path)

    return df

In [5]:
import os

os.chdir(r"/storage2/albert/iSTS-mgr/")
from src.data.data import get_dataframe
get_dataframe("headlines", "train", "chunked")

,sent1,sent2
0,"[ John Demjanjuk ] [ , ] [ convicted Nazi deat...",[ Former Nazi death camp guard Demjanjuk ] [ d...
1,[ Saudi Women ] [ Allowed ] [ To Compete ] [ A...,[ Saudis ] [ to permit ] [ women ] [ to compet...
2,[ Drone strike ] [ kills ] [ four suspected mi...,[ US drone strike ] [ kills ] [ 5 militants ] ...
3,[ Syria peace ] [ dashed ] [ as deadline passes ],"[ Syria peace plan conditions ] [ "" ] [ unacce..."
4,[ Syria ] [ blames ] [ rebels ] [ for Houla ma...,[ Syrian regime ] [ denies ] [ role ] [ in Hou...
...,...,...
751,[ SFG meeting ] [ reviews ] [ situation ] [ in...,[ US Senator McCain ] [ meets ] [ opposition l...
752,[ Tens ] [ of thousands ] [ line up ] [ to cas...,[ Tens ] [ of Thousands ] [ of Ukrainians ] [ ...
753,[ Obama ] [ to press ] [ Congress ] [ to act ]...,[ Obama ] [ pledges ] [ to reignite ] [ economy ]
754,[ Islamist parties ] [ in Egypt ] [ unite ] [ ...,[ Pak missions ] [ in UAE ] [ gear up ] [ for ...


In [12]:
df, labels = get_df(r"C:\Users\alber\Desktop\MGR\main_work\Data\train_data\wa\processed_answers-students.xml")


In [13]:
labels

0        EQUI
1        SIMI
2        EQUI
3       NOALI
4       NOALI
        ...  
1843    NOALI
1844     EQUI
1845    NOALI
1846     EQUI
1847     EQUI
Name: class_1, Length: 1848, dtype: object

In [1]:
possible_texts = ["SIMI", "EOS", "EQUI"]
text = "EQUItrain SIMIno capture"

# Iterate through possible texts and check if they are substrings in the text
for possible_word in possible_texts:
    if possible_word in text:
        print(f"First match: {possible_word}")
        break

First match: SIMI


In [2]:
possible_texts = ["SIMI", "EOS", "EQUI"]
text = "EQUItrain SIMIno capture"

# Dictionary to store the first occurrence index of each possible word
occurrences = {}

# Find the index of each possible word if it's in the text
for possible_word in possible_texts:
    index = text.find(possible_word)
    if index != -1:  # If the word is found in the text
        occurrences[possible_word] = index

# If we found any matches, return the one with the smallest index (first occurrence)
if occurrences:
    first_match = min(occurrences, key=occurrences.get)
    print(f"First match: {first_match}")
else:
    print("No match found.")

First match: EQUI


In [4]:
text = "siema 1 i co"

"1" in text

True

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct", torch_dtype="auto")
        # model = model.to("cuda")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B-Instruct")


In [3]:
model.generation_config

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

In [4]:
model.config

Qwen2Config {
  "_name_or_path": "Qwen/Qwen2-0.5B-Instruct",
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_theta": 1000000.0,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.44.2",
  "use_cache": true,
  "use_sliding_window": false,
  "vocab_size": 151936
}